## **Librerías**

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.options.display.max_columns = False

In [4]:
from datetime import datetime

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

## **Datos**

In [6]:
df_retail = pd.read_csv('../Data/online_retail.csv')

In [8]:
df_retail.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
377899,525871,22900,SET 2 TEA TOWELS I LOVE LONDON,2,2010-10-07 13:42:00,2.95,17354.0,United Kingdom


In [9]:
df_retail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      525461 non-null  object 
 1   StockCode    525461 non-null  object 
 2   Description  522533 non-null  object 
 3   Quantity     525461 non-null  int64  
 4   InvoiceDate  525461 non-null  object 
 5   Price        525461 non-null  float64
 6   Customer ID  417534 non-null  float64
 7   Country      525461 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.1+ MB


## **Feature Engineering**

### **Limpieza de Columnas**

In [10]:
# Ajustamos el nombre de algunas columnas
df_retail.rename(columns={
    'Customer ID': 'CustomerID'
}, inplace=True)

In [11]:
# Ajustamos el formato de nuestro id
df_retail['CustomerID'].fillna(0, inplace=True)
df_retail['CustomerID'] = df_retail['CustomerID'].apply(lambda x: int(x))

C:\Users\jmart\AppData\Local\Temp\ipykernel_16948\823382221.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_retail['CustomerID'].fillna(0, inplace=True)


In [12]:
# Ajustamos el formato de nuestra fecha de factura
df_retail['InvoiceDate'] = df_retail['InvoiceDate'].apply(lambda x: pd.to_datetime(x))

In [13]:
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


### **Creación de Features**

In [15]:
# Creamos una variable de mes de facturación
df_retail['InvoiceMonth'] = df_retail['InvoiceDate'].apply(lambda x: datetime(x.year, x.month, x.day))

In [16]:
# Agrupamos a nuestros usuarios por mes de facturación
grouping = df_retail.groupby('CustomerID')['InvoiceMonth']

In [17]:
# Nos quedamos con el primer registro defacturación disponible
df_retail['CohortMonth'] = grouping.transform('min')

In [18]:
df_retail.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,InvoiceMonth,CohortMonth
365132,524765,22029,SPACEBOY BIRTHDAY CARD,2,2010-09-30 15:47:00,0.85,0,United Kingdom,2010-09-30,2009-12-01


In [19]:
# Calculamos los días entre cada compra y la primera
df_retail['DateDiff'] = df_retail['InvoiceMonth'] - df_retail['CohortMonth']
df_retail['DateDiff'] = pd.to_numeric(df_retail['DateDiff'].dt.days, downcast='integer')
df_retail['DateDiff'] = df_retail['DateDiff'].apply(lambda x: int(x))

df_retail['DateDiffMonths'] = df_retail['DateDiff'].apply(lambda x: int(round(x/30) + 1))

In [20]:
df_retail['Cohort'] = df_retail['CohortMonth'].apply(lambda x: str(x)[:7])

In [21]:
df_retail.sample()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,InvoiceMonth,CohortMonth,DateDiff,DateDiffMonths,Cohort
36797,492420,84509A,ENGLISH ROSE PLACEMATS,1,2009-12-16 17:36:00,3.75,0,United Kingdom,2009-12-16,2009-12-01,15,1,2009-12


In [22]:
cohorts = pd.crosstab(
    index=df_retail['Cohort'], 
    columns=df_retail['DateDiffMonths']
)

In [23]:
cohorts.iloc[:,0]

Cohort
2009-12    40723
2010-01     9599
2010-02     9815
2010-03    11858
2010-04     6774
2010-05     6527
2010-06     7646
2010-07     5192
2010-08     4717
2010-09     6290
2010-10    11381
2010-11    11571
2010-12     1214
Name: 1, dtype: int64

In [24]:
cohorts

DateDiffMonths,1,2,3,4,5,6,7,8,9,10,11,12,13
Cohort,,,,,,,,,,,,,
2009-12,40723,16200,20995,21225,20023,20993,21373,19290,18105,16768,25705,29645,38378
2010-01,9599,2418,2712,2854,3367,2218,2653,2202,2860,3792,3628,1006,0
2010-02,9815,1614,2518,2585,2470,1900,1956,2645,3086,3294,964,0,0
2010-03,11858,2021,3129,3089,2581,2361,3012,3516,4420,1044,0,0,0
2010-04,6774,935,1068,985,1086,1385,2021,1861,516,0,0,0,0
2010-05,6527,856,689,690,657,2102,1765,570,0,0,0,0,0
2010-06,7646,1005,1151,1055,2454,2129,944,0,0,0,0,0,0
2010-07,5192,642,1127,1744,1787,711,0,0,0,0,0,0,0
2010-08,4717,1050,1208,1506,478,0,0,0,0,0,0,0,0


In [25]:
round(cohorts.divide(cohorts.iloc[:,0], axis=0)*100, 2)

DateDiffMonths,1,2,3,4,5,6,7,8,9,10,11,12,13
Cohort,,,,,,,,,,,,,
2009-12,100.0,39.78,51.56,52.12,49.17,51.55,52.48,47.37,44.46,41.18,63.12,72.80,94.24
2010-01,100.0,25.19,28.25,29.73,35.08,23.11,27.64,22.94,29.79,39.50,37.80,10.48,0.00
2010-02,100.0,16.44,25.65,26.34,25.17,19.36,19.93,26.95,31.44,33.56,9.82,0.00,0.00
2010-03,100.0,17.04,26.39,26.05,21.77,19.91,25.40,29.65,37.27,8.80,0.00,0.00,0.00
2010-04,100.0,13.80,15.77,14.54,16.03,20.45,29.83,27.47,7.62,0.00,0.00,0.00,0.00
2010-05,100.0,13.11,10.56,10.57,10.07,32.20,27.04,8.73,0.00,0.00,0.00,0.00,0.00
2010-06,100.0,13.14,15.05,13.80,32.10,27.84,12.35,0.00,0.00,0.00,0.00,0.00,0.00
2010-07,100.0,12.37,21.71,33.59,34.42,13.69,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2010-08,100.0,22.26,25.61,31.93,10.13,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
